# Multilevel merged-cell failure extraction

Interactive Napari workflow for collecting two independently timed parent-cell states and one later merged-cell state as a single diagnostic case.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import pandas as pd
import sys

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root whether Jupyter starts at the root or notebook folder."""

    start = (start or Path.cwd()).resolve()

    for candidate in (start, *start.parents):
        has_data = (candidate / "data").exists()
        has_project_code = (
            (candidate / "diagnostics").exists()
            or (candidate / "notebooks").exists()
        )

        if has_data and has_project_code:
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Start Jupyter inside the repository "
        "or set PROJECT_ROOT manually."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_ROOT = PROJECT_ROOT / "data" / "sample"

SAMPLE_ID = "44b6_0113de3b"

PROCESSED_DIR = (
    DATA_ROOT
    / "processed"
    / "stage_6_processed_dataset"
    / SAMPLE_ID
)

CELLS_DIR = PROCESSED_DIR / "cells"

print("Project root:", PROJECT_ROOT)
print("Sample:", SAMPLE_ID)


In [ ]:
cell_files = sorted(CELLS_DIR.glob("t*.csv"))

time_frames = [
    pd.read_csv(file)
    for file in cell_files
]

print(f"Loaded {len(time_frames)} timepoints.")

In [ ]:
import json
from pathlib import Path

import pandas as pd


TRACKING_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "sample"
    / "processed"
    / "stage_8_track_stitching"
)

# Change this variable if the tracking stage is stored elsewhere.
output_dir = TRACKING_OUTPUT_DIR

# ------------------------------------------------------------
# Load detections
# ------------------------------------------------------------
detections_df = pd.read_csv(output_dir / "detections.csv")

time_frames = [
    frame_df.reset_index(drop=True)
    for _, frame_df in detections_df.groupby("frame")
]

# ------------------------------------------------------------
# Load tracks and diagnostics
# ------------------------------------------------------------
tracks = pd.read_csv(output_dir / "tracks.csv")
segmentation_events = pd.read_csv(output_dir / "segmentation_events.csv")
track_endings = pd.read_csv(output_dir / "track_endings.csv")

with (output_dir / "metadata.json").open("r", encoding="utf-8") as file:
    metadata = json.load(file)

print("Loaded all tracking and stitching results from:", output_dir)


Approximately assign cell id

In [ ]:
import numpy as np
from scipy.spatial import cKDTree

# Reload the original per-frame cell files (these have cell_id + centroids)
cell_files = sorted(CELLS_DIR.glob("t*.csv"))

cell_frames = []
for i, file in enumerate(cell_files):
    df = pd.read_csv(file)
    df["frame"] = i  # verify this matches the frame numbering used in tracks.csv
    cell_frames.append(df)

cells_df = pd.concat(cell_frames, ignore_index=True)

# Nearest-neighbor match each track row to its source cell, per frame
tracks = tracks.copy()
tracks["cell_id"] = -1

for frame, frame_tracks in tracks.groupby("frame"):
    frame_cells = cells_df[cells_df["frame"] == frame]
    if frame_cells.empty:
        continue
    tree = cKDTree(frame_cells[["centroid_z", "centroid_y", "centroid_x"]].to_numpy())
    dist, idx = tree.query(frame_tracks[["z", "y", "x"]].to_numpy())
    tracks.loc[frame_tracks.index, "cell_id"] = frame_cells["cell_id"].to_numpy()[idx]

# sanity check — distances should be ~0 (or very small) if the match is correct
print("max match distance:", dist.max() if len(dist) else "n/a")
print("unmatched rows:", (tracks["cell_id"] == -1).sum())

In [ ]:
import zarr

ZARR_PATH = (
        DATA_ROOT
        / "biohub_5samples_20timepoints"
        / "train"
        / SAMPLE_ID
        / f"{SAMPLE_ID}.zarr"
)

ARRAY_PATH = ZARR_PATH / "0"

original_volume = zarr.open_array(
    str(ARRAY_PATH),
    mode="r"
)

In [ ]:
# ------------------------------------------------------------
# Convert tracks to Napari format
# ------------------------------------------------------------

tracks_array = tracks[
    ["track_id", "frame", "z", "y", "x"]
].to_numpy(dtype=float)

points_array = tracks[
    ["frame", "z", "y", "x"]
].to_numpy(dtype=float)

track_ids = tracks["track_id"].to_numpy()

Load pre-processed data

In [ ]:
from pathlib import Path

import dask.array as da
import numpy as np


PREPROCESSING_DIR = PROCESSED_DIR / "preprocessing"
MASKING_DIR = PROCESSED_DIR / "masking"
SEGMENTATION_DIR = PROCESSED_DIR / "segmentation"


def load_npy_time_series(
        directory: Path,
        expected_frames: int | None = None,
) -> tuple[da.Array, list[Path]]:
    """
    Load per-frame 3D .npy files as one lazy 4D Dask array.

    Expected files:
        t000.npy
        t001.npy
        ...

    Returned shape:
        (T, Z, Y, X)
    """

    files = sorted(directory.glob("t*.npy"))

    if not files:
        raise FileNotFoundError(
            f"No time-point arrays found in:\n{directory}"
        )

    if expected_frames is not None and len(files) != expected_frames:
        raise ValueError(
            f"{directory.name}: expected {expected_frames} frames, "
            f"but found {len(files)}."
        )

    first = np.load(
        files[0],
        mmap_mode="r",
        allow_pickle=False,
    )

    if first.ndim != 3:
        raise ValueError(
            f"Expected 3D arrays in {directory}, "
            f"but {files[0].name} has shape {first.shape}."
        )

    spatial_shape = first.shape

    # Small chunks allow the extractor to read only the selected crop.
    chunks = (
        min(8, spatial_shape[0]),
        min(64, spatial_shape[1]),
        min(64, spatial_shape[2]),
    )

    arrays = []

    for path in files:
        array = np.load(
            path,
            mmap_mode="r",
            allow_pickle=False,
        )

        if array.shape != spatial_shape:
            raise ValueError(
                f"Inconsistent shape in {path.name}: "
                f"expected {spatial_shape}, found {array.shape}."
            )

        arrays.append(
            da.from_array(
                array,
                chunks=chunks,
            )
        )

    sequence = da.stack(
        arrays,
        axis=0,
    )

    return sequence, files

In [ ]:
NUM_TIMEPOINTS = original_volume.shape[0]

preprocessed_volume, preprocessing_files = load_npy_time_series(
    PREPROCESSING_DIR,
    expected_frames=NUM_TIMEPOINTS,
)

binary_mask_volume, masking_files = load_npy_time_series(
    MASKING_DIR,
    expected_frames=NUM_TIMEPOINTS,
)

instance_labels_volume, segmentation_files = load_npy_time_series(
    SEGMENTATION_DIR,
    expected_frames=NUM_TIMEPOINTS,
)

In [ ]:
print(
    "Raw:",
    original_volume.shape,
    original_volume.dtype,
)

print(
    "Preprocessed:",
    preprocessed_volume.shape,
    preprocessed_volume.dtype,
)

print(
    "Binary mask:",
    binary_mask_volume.shape,
    binary_mask_volume.dtype,
)

print(
    "Instance labels:",
    instance_labels_volume.shape,
    instance_labels_volume.dtype,
)

**Broken Tracks**

Red cells -> Tracks that will break before the last frame

Green cells -> Tracks that will born after the first frame

In [ ]:
# Endpoint cells this close to any spatial boundary are treated
# as entering/leaving the imaging volume rather than tracking failures.
BOUNDARY_MARGIN_UM = 4.0

# Whether boundary-entry and boundary-exit tracks should be added
# to Napari as separate diagnostic layers.
SHOW_BOUNDARY_TRACKS = False

## Multilevel merged-cell case extraction

This notebook records each biological state from the **current Napari timepoint**, so the two source cells do not need to come from the same frame.

Workflow:

1. Choose one shared `(Z, Y, X)` box size.
2. Move to the frame containing **cell 1**, enter its frame-local ID, preview the box, and click **Save cell 1**.
3. Move to the frame containing **cell 2**, enter its frame-local ID, preview, and click **Save cell 2**.
4. Move to the later frame containing the **merged cell**, enter its ID, preview, and click **Save complete case**.

The first captured parent locks the shared box size until **Reset case** is clicked. Parent captures are held in memory, and the final button writes the complete case atomically.

Output layout:

```text
data/extracted/merged_cells/<case_name>/
├── metadata.json
├── cells.csv
├── cell_1/
├── cell_2/
└── merged_cell/
```

Each role directory contains aligned raw, preprocessed, binary-mask, instance-label, selected-cell-mask, masked-intensity, feature, and metadata files. The three crops have the same shape but retain independent global bounds and frames.


In [ ]:
import json
import shutil
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
from napari.utils.notifications import show_error, show_info
from qtpy.QtCore import Qt
from qtpy.QtWidgets import (
    QCheckBox,
    QFormLayout,
    QGridLayout,
    QGroupBox,
    QHBoxLayout,
    QLabel,
    QPushButton,
    QSpinBox,
    QVBoxLayout,
    QWidget,
)


# ------------------------------------------------------------
# Multilevel merged-cell extraction configuration
# ------------------------------------------------------------

MERGED_CELLS_DIR = (
    PROJECT_ROOT
    / "data"
    / "extracted"
    / "merged_cells"
)

DEFAULT_MERGED_CASE_BOX_SIZE = (12, 50, 50)  # (Z, Y, X), voxels
MERGED_CASE_PAD_VALUE = 0
MERGED_CASE_PREVIEW_LAYER = "Merged-case extraction preview"


# ------------------------------------------------------------
# General helpers
# ------------------------------------------------------------

def _json_compatible(value: Any) -> Any:
    """Recursively convert NumPy, pandas, and Path values for JSON."""

    if isinstance(value, Path):
        return str(value)

    if isinstance(value, Mapping):
        return {
            str(key): _json_compatible(item)
            for key, item in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [_json_compatible(item) for item in value]

    if isinstance(value, np.ndarray):
        return value.tolist()

    if isinstance(value, np.generic):
        return value.item()

    if value is None:
        return None

    try:
        if bool(pd.isna(value)):
            return None
    except (TypeError, ValueError):
        pass

    return value


def _write_json(path: Path, data: Mapping[str, Any]) -> None:
    with path.open("w", encoding="utf-8") as file:
        json.dump(
            _json_compatible(dict(data)),
            file,
            indent=2,
            allow_nan=False,
        )


def _validate_box_size(box_size: Sequence[int]) -> tuple[int, int, int]:
    size = np.asarray(box_size, dtype=int)

    if size.shape != (3,):
        raise ValueError(
            "box_size must contain exactly three values in (Z, Y, X) order."
        )

    if np.any(size <= 0):
        raise ValueError(f"Box dimensions must be positive; received {tuple(size)}.")

    return tuple(int(value) for value in size)


def _require_cell_row(
    cells: pd.DataFrame,
    frame: int,
    cell_id: int,
) -> pd.Series:
    """Return one frame-local Stage-6 cell row."""

    required_columns = {
        "frame",
        "cell_id",
        "centroid_z",
        "centroid_y",
        "centroid_x",
    }
    missing = required_columns.difference(cells.columns)

    if missing:
        raise KeyError(
            f"cells_df is missing required columns: {sorted(missing)}"
        )

    rows = cells.loc[
        (cells["frame"].astype(int) == int(frame))
        & (cells["cell_id"].astype(int) == int(cell_id))
    ]

    if rows.empty:
        frame_ids = (
            cells.loc[cells["frame"].astype(int) == int(frame), "cell_id"]
            .astype(int)
            .sort_values()
            .drop_duplicates()
            .tolist()
        )
        preview = frame_ids[:30]
        suffix = " ..." if len(frame_ids) > 30 else ""
        raise ValueError(
            f"Cell {cell_id} was not found in frame {frame}. "
            f"Available IDs: {preview}{suffix}"
        )

    if len(rows) > 1:
        raise ValueError(
            f"Found {len(rows)} rows for cell {cell_id} in frame {frame}; "
            "expected exactly one."
        )

    return rows.iloc[0].copy()


def _calculate_box_bounds(
    centroid_zyx: Sequence[float],
    box_size_zyx: Sequence[int],
    spatial_shape_zyx: Sequence[int],
) -> dict[str, list[int]]:
    """Calculate fixed-size crop bounds and any boundary padding."""

    size = np.asarray(_validate_box_size(box_size_zyx), dtype=int)
    centroid = np.asarray(centroid_zyx, dtype=float)
    spatial_shape = np.asarray(spatial_shape_zyx, dtype=int)

    if centroid.shape != (3,):
        raise ValueError("centroid_zyx must contain (Z, Y, X).")

    if spatial_shape.shape != (3,) or np.any(spatial_shape <= 0):
        raise ValueError("spatial_shape_zyx must contain three positive values.")

    centroid_index = np.rint(centroid).astype(int)
    requested_start = centroid_index - size // 2
    requested_stop = requested_start + size

    clipped_start = np.maximum(requested_start, 0)
    clipped_stop = np.minimum(requested_stop, spatial_shape)

    if np.any(clipped_start >= clipped_stop):
        raise ValueError(
            "The requested crop does not overlap the image volume. "
            f"Centroid index: {centroid_index.tolist()}"
        )

    padding_before = np.maximum(-requested_start, 0)
    padding_after = np.maximum(requested_stop - spatial_shape, 0)

    return {
        "centroid_index_zyx": centroid_index.tolist(),
        "requested_start_zyx": requested_start.tolist(),
        "requested_stop_zyx": requested_stop.tolist(),
        "clipped_start_zyx": clipped_start.tolist(),
        "clipped_stop_zyx": clipped_stop.tolist(),
        "padding_before_zyx": padding_before.tolist(),
        "padding_after_zyx": padding_after.tolist(),
    }


def _extract_aligned_box(
    volume: Any,
    *,
    frame: int,
    box_size_zyx: Sequence[int],
    bounds: Mapping[str, Sequence[int]],
    pad_value: int | float | bool = 0,
) -> np.ndarray:
    """Extract one fixed-size crop without materializing a complete frame."""

    if volume is None:
        raise ValueError("The requested source volume is not available.")

    if not hasattr(volume, "shape") or len(volume.shape) != 4:
        raise ValueError(
            "Expected an aligned source with shape (T, Z, Y, X); "
            f"found {getattr(volume, 'shape', None)}."
        )

    frame = int(frame)
    if not 0 <= frame < int(volume.shape[0]):
        raise IndexError(
            f"Frame {frame} is outside 0 to {int(volume.shape[0]) - 1}."
        )

    size = np.asarray(_validate_box_size(box_size_zyx), dtype=int)
    requested_start = np.asarray(bounds["requested_start_zyx"], dtype=int)
    clipped_start = np.asarray(bounds["clipped_start_zyx"], dtype=int)
    clipped_stop = np.asarray(bounds["clipped_stop_zyx"], dtype=int)

    source_slices = tuple(
        slice(int(start), int(stop))
        for start, stop in zip(clipped_start, clipped_stop)
    )

    destination_start = clipped_start - requested_start
    destination_stop = destination_start + clipped_stop - clipped_start
    destination_slices = tuple(
        slice(int(start), int(stop))
        for start, stop in zip(destination_start, destination_stop)
    )

    crop = np.full(
        tuple(int(value) for value in size),
        pad_value,
        dtype=volume.dtype,
    )

    source = volume[(frame, *source_slices)]
    if hasattr(source, "compute"):
        source = source.compute()

    crop[destination_slices] = np.asarray(source)
    return crop


def _make_box_wireframe(
    frame: int,
    requested_start_zyx: Sequence[int],
    requested_stop_zyx: Sequence[int],
) -> list[np.ndarray]:
    """Create 12 Napari line segments around one voxel-aligned 3D box."""

    start = np.asarray(requested_start_zyx, dtype=float) - 0.5
    stop = np.asarray(requested_stop_zyx, dtype=float) - 0.5

    z0, y0, x0 = start
    z1, y1, x1 = stop

    corners = np.asarray(
        [
            [z0, y0, x0],
            [z0, y0, x1],
            [z0, y1, x0],
            [z0, y1, x1],
            [z1, y0, x0],
            [z1, y0, x1],
            [z1, y1, x0],
            [z1, y1, x1],
        ],
        dtype=float,
    )

    edge_pairs = (
        (0, 1), (0, 2), (1, 3), (2, 3),
        (4, 5), (4, 6), (5, 7), (6, 7),
        (0, 4), (1, 5), (2, 6), (3, 7),
    )

    return [
        np.asarray(
            [
                [float(frame), *corners[first]],
                [float(frame), *corners[second]],
            ],
            dtype=float,
        )
        for first, second in edge_pairs
    ]


def _mask_bbox(mask: np.ndarray) -> tuple[list[int], list[int]]:
    coordinates = np.argwhere(mask)
    if coordinates.size == 0:
        raise ValueError("The selected instance mask is empty inside the crop.")

    start = coordinates.min(axis=0).astype(int).tolist()
    stop = (coordinates.max(axis=0) + 1).astype(int).tolist()
    return start, stop


def _source_at_frame(
    paths: Sequence[Path] | None,
    frame: int,
) -> str | None:
    if paths is None or not 0 <= int(frame) < len(paths):
        return None
    return str(Path(paths[int(frame)]))


# ------------------------------------------------------------
# Capture and persistence
# ------------------------------------------------------------

def capture_cell_state(
    *,
    role: str,
    frame: int,
    cell_id: int,
    box_size_zyx: Sequence[int],
    cells: pd.DataFrame,
    raw_volume: Any,
    preprocessed_volume: Any,
    binary_mask_volume: Any,
    instance_labels_volume: Any,
    voxel_size_zyx: Sequence[float],
    source_cell_files: Sequence[Path] | None = None,
    source_raw_zarr: Path | None = None,
    source_preprocessed_files: Sequence[Path] | None = None,
    source_binary_mask_files: Sequence[Path] | None = None,
    source_instance_label_files: Sequence[Path] | None = None,
) -> dict[str, Any]:
    """Materialize one role at its independently selected frame."""

    frame = int(frame)
    cell_id = int(cell_id)
    box_size = _validate_box_size(box_size_zyx)

    if cell_id <= 0:
        raise ValueError("Cell IDs must be positive instance labels.")

    row = _require_cell_row(cells, frame=frame, cell_id=cell_id)
    centroid_zyx = row[
        ["centroid_z", "centroid_y", "centroid_x"]
    ].to_numpy(dtype=float)

    spatial_shape = tuple(int(value) for value in raw_volume.shape[1:])
    bounds = _calculate_box_bounds(
        centroid_zyx=centroid_zyx,
        box_size_zyx=box_size,
        spatial_shape_zyx=spatial_shape,
    )

    raw_crop = _extract_aligned_box(
        raw_volume,
        frame=frame,
        box_size_zyx=box_size,
        bounds=bounds,
        pad_value=MERGED_CASE_PAD_VALUE,
    )
    preprocessed_crop = _extract_aligned_box(
        preprocessed_volume,
        frame=frame,
        box_size_zyx=box_size,
        bounds=bounds,
        pad_value=MERGED_CASE_PAD_VALUE,
    )
    binary_crop = _extract_aligned_box(
        binary_mask_volume,
        frame=frame,
        box_size_zyx=box_size,
        bounds=bounds,
        pad_value=False,
    ).astype(bool, copy=False)
    labels_crop = _extract_aligned_box(
        instance_labels_volume,
        frame=frame,
        box_size_zyx=box_size,
        bounds=bounds,
        pad_value=0,
    )

    selected_mask = labels_crop == cell_id
    if not selected_mask.any():
        raise ValueError(
            f"Instance label {cell_id} is absent from its extracted box at "
            f"frame {frame}. Check the cell table and segmentation labels."
        )

    bbox_start, bbox_stop = _mask_bbox(selected_mask)
    requested_start = np.asarray(bounds["requested_start_zyx"], dtype=float)
    centroid_crop_zyx = centroid_zyx - requested_start

    artifacts = {
        "raw": raw_crop,
        "preprocessed": preprocessed_crop,
        "binary_mask": binary_crop.astype(np.uint8),
        "instance_labels": labels_crop,
        "selected_cell_mask": selected_mask.astype(np.uint8),
        "raw_masked": np.where(selected_mask, raw_crop, 0),
        "preprocessed_masked": np.where(selected_mask, preprocessed_crop, 0),
    }

    source_files = {
        "cell_table": _source_at_frame(source_cell_files, frame),
        "raw_zarr_array": str(source_raw_zarr) if source_raw_zarr else None,
        "preprocessed": _source_at_frame(source_preprocessed_files, frame),
        "binary_mask": _source_at_frame(source_binary_mask_files, frame),
        "instance_labels": _source_at_frame(source_instance_label_files, frame),
    }

    metadata = {
        "role": str(role),
        "frame": frame,
        "cell_id": cell_id,
        "centroid_global_zyx": centroid_zyx,
        "centroid_crop_zyx": centroid_crop_zyx,
        "box_size_zyx": box_size,
        "voxel_size_zyx": tuple(float(value) for value in voxel_size_zyx),
        "bounds": bounds,
        "selected_cell_voxels": int(selected_mask.sum()),
        "selected_cell_bbox_crop_start_zyx": bbox_start,
        "selected_cell_bbox_crop_stop_zyx": bbox_stop,
        "source_files": {
            name: path
            for name, path in source_files.items()
            if path is not None
        },
        "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    return {
        "role": str(role),
        "frame": frame,
        "cell_id": cell_id,
        "box_size_zyx": box_size,
        "feature_row": row,
        "metadata": metadata,
        "artifacts": artifacts,
    }


def _save_role_directory(
    role_dir: Path,
    capture: Mapping[str, Any],
) -> None:
    role_dir.mkdir(parents=True, exist_ok=False)

    artifact_files: dict[str, str] = {}
    for name, array in capture["artifacts"].items():
        filename = f"{name}.npy"
        np.save(role_dir / filename, np.asarray(array), allow_pickle=False)
        artifact_files[name] = filename

    feature_row = capture["feature_row"]
    pd.DataFrame([feature_row]).to_csv(
        role_dir / "features.csv",
        index=False,
    )

    role_metadata = dict(capture["metadata"])
    role_metadata["artifact_files"] = artifact_files
    role_metadata["features_file"] = "features.csv"
    _write_json(role_dir / "metadata.json", role_metadata)


def _update_merged_case_manifest(
    merged_cells_dir: Path,
    record: Mapping[str, Any],
) -> None:
    merged_cells_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = merged_cells_dir / "manifest.csv"
    new_row = pd.DataFrame([dict(record)])

    if manifest_path.exists():
        manifest = pd.concat(
            [pd.read_csv(manifest_path), new_row],
            ignore_index=True,
        )
    else:
        manifest = new_row

    if "case_name" in manifest.columns:
        manifest = manifest.drop_duplicates(
            subset=["case_name"],
            keep="last",
        )

    sort_columns = [
        column
        for column in ["sample_id", "merged_frame", "merged_cell_id"]
        if column in manifest.columns
    ]
    if sort_columns:
        manifest = manifest.sort_values(sort_columns)

    manifest.to_csv(manifest_path, index=False)


def save_multilevel_merged_cell_case(
    *,
    sample_id: str,
    cell_1: Mapping[str, Any],
    cell_2: Mapping[str, Any],
    merged_cell: Mapping[str, Any],
    output_dir: Path = MERGED_CELLS_DIR,
    overwrite: bool = False,
) -> Path:
    """Atomically save three independently timed, equally sized cell crops."""

    captures = {
        "cell_1": cell_1,
        "cell_2": cell_2,
        "merged_cell": merged_cell,
    }

    box_sizes = {
        tuple(int(value) for value in capture["box_size_zyx"])
        for capture in captures.values()
    }
    if len(box_sizes) != 1:
        raise ValueError(
            "All three captures must use the same box size. Reset and capture "
            "the case again with one shared box size."
        )

    cell_1_key = (int(cell_1["frame"]), int(cell_1["cell_id"]))
    cell_2_key = (int(cell_2["frame"]), int(cell_2["cell_id"]))
    if cell_1_key == cell_2_key:
        raise ValueError("Cell 1 and cell 2 cannot be the same frame-local cell.")

    merged_frame = int(merged_cell["frame"])
    if int(cell_1["frame"]) >= merged_frame:
        raise ValueError("Cell 1 must be captured from a frame before the merged cell.")
    if int(cell_2["frame"]) >= merged_frame:
        raise ValueError("Cell 2 must be captured from a frame before the merged cell.")

    case_name = (
        f"{sample_id}__"
        f"cell1_t{int(cell_1['frame']):03d}_id{int(cell_1['cell_id']):05d}__"
        f"cell2_t{int(cell_2['frame']):03d}_id{int(cell_2['cell_id']):05d}__"
        f"merged_t{merged_frame:03d}_id{int(merged_cell['cell_id']):05d}"
    )

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    case_dir = output_dir / case_name
    temporary_dir = output_dir / f".{case_name}.tmp"

    if case_dir.exists() and not overwrite:
        raise FileExistsError(
            f"This merged-cell case already exists:\n{case_dir}\n"
            "Enable overwrite or change one of the selected frame/cell pairs."
        )

    if temporary_dir.exists():
        shutil.rmtree(temporary_dir)
    temporary_dir.mkdir(parents=False, exist_ok=False)

    try:
        for role, capture in captures.items():
            _save_role_directory(temporary_dir / role, capture)

        box_size = next(iter(box_sizes))
        case_metadata = {
            "schema_version": 2,
            "case_name": case_name,
            "sample_id": str(sample_id),
            "created_at_utc": datetime.now(timezone.utc).isoformat(),
            "box_size_zyx": box_size,
            "voxel_size_zyx": merged_cell["metadata"]["voxel_size_zyx"],
            "cell_1_frame": int(cell_1["frame"]),
            "cell_1_id": int(cell_1["cell_id"]),
            "cell_2_frame": int(cell_2["frame"]),
            "cell_2_id": int(cell_2["cell_id"]),
            "merged_frame": merged_frame,
            "merged_cell_id": int(merged_cell["cell_id"]),
            "roles": {
                role: {
                    "directory": role,
                    "frame": int(capture["frame"]),
                    "cell_id": int(capture["cell_id"]),
                    "centroid_global_zyx": capture["metadata"][
                        "centroid_global_zyx"
                    ],
                    "centroid_crop_zyx": capture["metadata"][
                        "centroid_crop_zyx"
                    ],
                    "bounds": capture["metadata"]["bounds"],
                }
                for role, capture in captures.items()
            },
            # Compatibility fields for analysis code that treats the merged
            # observation as the primary case.
            "frame": merged_frame,
            "cell_id": int(merged_cell["cell_id"]),
            "centroid_zyx": merged_cell["metadata"]["centroid_global_zyx"],
        }
        _write_json(temporary_dir / "metadata.json", case_metadata)

        case_rows = []
        for role, capture in captures.items():
            case_rows.append(
                {
                    "role": role,
                    "frame": int(capture["frame"]),
                    "cell_id": int(capture["cell_id"]),
                    "directory": role,
                }
            )
        pd.DataFrame(case_rows).to_csv(
            temporary_dir / "cells.csv",
            index=False,
        )

        if case_dir.exists():
            shutil.rmtree(case_dir)
        temporary_dir.rename(case_dir)

    except Exception:
        if temporary_dir.exists():
            shutil.rmtree(temporary_dir)
        raise

    _update_merged_case_manifest(
        output_dir,
        {
            "case_name": case_name,
            "sample_id": str(sample_id),
            "cell_1_frame": int(cell_1["frame"]),
            "cell_1_id": int(cell_1["cell_id"]),
            "cell_2_frame": int(cell_2["frame"]),
            "cell_2_id": int(cell_2["cell_id"]),
            "merged_frame": merged_frame,
            "merged_cell_id": int(merged_cell["cell_id"]),
            "box_depth": int(box_size[0]),
            "box_height": int(box_size[1]),
            "box_width": int(box_size[2]),
            "case_directory": str(case_dir),
            "metadata_path": str(case_dir / "metadata.json"),
        },
    )

    return case_dir


# ------------------------------------------------------------
# Napari widget
# ------------------------------------------------------------

class MultilevelMergedCellExtractorWidget(QWidget):
    """Capture two parent states and one later merged state from the viewer."""

    def __init__(
        self,
        *,
        viewer: Any,
        cells: pd.DataFrame,
        raw_volume: Any,
        preprocessed_volume: Any,
        binary_mask_volume: Any,
        instance_labels_volume: Any,
        sample_id: str,
        output_dir: Path = MERGED_CELLS_DIR,
        voxel_size_zyx: Sequence[float] = (1.625, 0.40625, 0.40625),
        default_box_size: Sequence[int] = DEFAULT_MERGED_CASE_BOX_SIZE,
        source_cell_files: Sequence[Path] | None = None,
        source_raw_zarr: Path | None = None,
        source_preprocessed_files: Sequence[Path] | None = None,
        source_binary_mask_files: Sequence[Path] | None = None,
        source_instance_label_files: Sequence[Path] | None = None,
        time_axis: int = 0,
        parent: QWidget | None = None,
    ) -> None:
        super().__init__(parent)

        self.viewer = viewer
        self.cells = cells
        self.raw_volume = raw_volume
        self.preprocessed_volume = preprocessed_volume
        self.binary_mask_volume = binary_mask_volume
        self.instance_labels_volume = instance_labels_volume
        self.sample_id = str(sample_id)
        self.output_dir = Path(output_dir)
        self.voxel_size_zyx = tuple(float(value) for value in voxel_size_zyx)
        self.time_axis = int(time_axis)

        self.source_cell_files = source_cell_files
        self.source_raw_zarr = source_raw_zarr
        self.source_preprocessed_files = source_preprocessed_files
        self.source_binary_mask_files = source_binary_mask_files
        self.source_instance_label_files = source_instance_label_files

        self._validate_sources()
        self._captures: dict[str, dict[str, Any]] = {}
        self._locked_box_size: tuple[int, int, int] | None = None
        self._preview_layer = None
        self._preview_role: str | None = None
        self._last_saved_dir: Path | None = None

        self._build_ui(_validate_box_size(default_box_size))
        self._connect_events()
        self._update_frame_label()
        self._set_status(
            "Select cell 1 in its frame, then cell 2 in its frame, then move "
            "to the merged frame and save the complete case."
        )

    def _validate_sources(self) -> None:
        reference_shape = tuple(int(value) for value in self.raw_volume.shape)
        if len(reference_shape) != 4:
            raise ValueError(
                f"raw_volume must have shape (T, Z, Y, X); found {reference_shape}."
            )

        for name, volume in {
            "preprocessed_volume": self.preprocessed_volume,
            "binary_mask_volume": self.binary_mask_volume,
            "instance_labels_volume": self.instance_labels_volume,
        }.items():
            shape = tuple(int(value) for value in volume.shape)
            if shape != reference_shape:
                raise ValueError(
                    f"{name} has shape {shape}, but raw_volume has "
                    f"shape {reference_shape}."
                )

        if len(self.voxel_size_zyx) != 3:
            raise ValueError("voxel_size_zyx must contain (Z, Y, X).")

    # --------------------------------------------------------
    # UI
    # --------------------------------------------------------

    def _build_ui(self, default_box_size: tuple[int, int, int]) -> None:
        layout = QVBoxLayout(self)

        title = QLabel("Multilevel merged-cell extractor")
        title.setStyleSheet("font-weight: bold;")
        layout.addWidget(title)

        instruction = QLabel(
            "The current Napari timepoint is recorded independently for every "
            "saved role. The first saved role locks one shared box size for the "
            "whole case."
        )
        instruction.setWordWrap(True)
        layout.addWidget(instruction)

        self.frame_label = QLabel()
        self.frame_label.setTextInteractionFlags(Qt.TextSelectableByMouse)
        layout.addWidget(self.frame_label)

        box_group = QGroupBox("Shared extraction box")
        box_form = QFormLayout(box_group)
        self.z_size_spin = self._make_size_spin(default_box_size[0])
        self.y_size_spin = self._make_size_spin(default_box_size[1])
        self.x_size_spin = self._make_size_spin(default_box_size[2])
        box_form.addRow("Depth Z", self.z_size_spin)
        box_form.addRow("Height Y", self.y_size_spin)
        box_form.addRow("Width X", self.x_size_spin)
        layout.addWidget(box_group)

        role_group = QGroupBox("Case roles")
        role_grid = QGridLayout(role_group)
        role_grid.addWidget(QLabel("Role"), 0, 0)
        role_grid.addWidget(QLabel("Cell ID"), 0, 1)
        role_grid.addWidget(QLabel("Preview"), 0, 2)
        role_grid.addWidget(QLabel("Capture / save"), 0, 3)

        self.role_id_spins: dict[str, QSpinBox] = {}
        self.role_status_labels: dict[str, QLabel] = {}

        role_specs = [
            ("cell_1", "Cell 1", "Save cell 1"),
            ("cell_2", "Cell 2", "Save cell 2"),
            ("merged_cell", "Merged cell", "Save complete case"),
        ]

        for row_index, (role, label, action_text) in enumerate(role_specs, start=1):
            role_grid.addWidget(QLabel(label), row_index, 0)

            cell_spin = QSpinBox()
            cell_spin.setRange(1, 2_147_483_647)
            cell_spin.setKeyboardTracking(False)
            cell_spin.setValue(1 if role != "cell_2" else 2)
            self.role_id_spins[role] = cell_spin
            role_grid.addWidget(cell_spin, row_index, 1)

            preview_button = QPushButton("Preview box")
            preview_button.clicked.connect(
                lambda _checked=False, selected_role=role: self.preview_role(
                    selected_role
                )
            )
            role_grid.addWidget(preview_button, row_index, 2)

            action_button = QPushButton(action_text)
            if role == "merged_cell":
                action_button.clicked.connect(self.save_complete_case)
            else:
                action_button.clicked.connect(
                    lambda _checked=False, selected_role=role: self.capture_parent(
                        selected_role
                    )
                )
            role_grid.addWidget(action_button, row_index, 3)

            status_label = QLabel("Not captured")
            status_label.setWordWrap(True)
            status_label.setTextInteractionFlags(Qt.TextSelectableByMouse)
            self.role_status_labels[role] = status_label
            role_grid.addWidget(status_label, row_index + 3, 0, 1, 4)

        layout.addWidget(role_group)

        self.overwrite_checkbox = QCheckBox("Overwrite an existing identical case")
        self.overwrite_checkbox.setChecked(False)
        layout.addWidget(self.overwrite_checkbox)

        button_row = QHBoxLayout()
        self.clear_preview_button = QPushButton("Clear preview")
        self.reset_button = QPushButton("Reset case")
        button_row.addWidget(self.clear_preview_button)
        button_row.addWidget(self.reset_button)
        layout.addLayout(button_row)

        output_label = QLabel(f"Output: {self.output_dir}")
        output_label.setWordWrap(True)
        output_label.setTextInteractionFlags(Qt.TextSelectableByMouse)
        layout.addWidget(output_label)

        self.status_label = QLabel()
        self.status_label.setWordWrap(True)
        self.status_label.setTextInteractionFlags(Qt.TextSelectableByMouse)
        layout.addWidget(self.status_label)
        layout.addStretch(1)

    @staticmethod
    def _make_size_spin(value: int) -> QSpinBox:
        spin = QSpinBox()
        spin.setRange(1, 100_001)
        spin.setSingleStep(2)
        spin.setKeyboardTracking(False)
        spin.setValue(int(value))
        return spin

    def _connect_events(self) -> None:
        self.clear_preview_button.clicked.connect(self.clear_preview)
        self.reset_button.clicked.connect(self.reset_case)

        for role, spin in self.role_id_spins.items():
            spin.valueChanged.connect(
                lambda _value, selected_role=role: self._on_role_id_changed(
                    selected_role
                )
            )

        try:
            self.viewer.dims.events.current_step.connect(self._on_frame_changed)
        except AttributeError:
            pass

    # --------------------------------------------------------
    # Current state and preview
    # --------------------------------------------------------

    def current_frame(self) -> int:
        return int(round(self.viewer.dims.current_step[self.time_axis]))

    def current_box_size(self) -> tuple[int, int, int]:
        size = (
            self.z_size_spin.value(),
            self.y_size_spin.value(),
            self.x_size_spin.value(),
        )
        return _validate_box_size(size)

    def _update_frame_label(self) -> None:
        frame = self.current_frame()
        maximum = int(self.raw_volume.shape[0]) - 1
        self.frame_label.setText(f"Current frame: {frame} / {maximum}")

    def _on_frame_changed(self, _event: Any = None) -> None:
        self._update_frame_label()
        self.clear_preview(update_status=False)

    def _on_role_id_changed(self, role: str) -> None:
        self.clear_preview(update_status=False)

        if role in self._captures:
            del self._captures[role]
            self.role_status_labels[role].setText("Changed — capture again")

            if role in {"cell_1", "cell_2"} and not any(
                parent_role in self._captures
                for parent_role in ("cell_1", "cell_2")
            ):
                self._unlock_box_size()

    def preview_role(self, role: str) -> None:
        try:
            self._update_frame_label()
            frame = self.current_frame()
            cell_id = int(self.role_id_spins[role].value())
            box_size = self.current_box_size()

            if self._locked_box_size is not None and box_size != self._locked_box_size:
                raise ValueError(
                    f"This case is locked to box size {self._locked_box_size}. "
                    "Reset the case to choose another size."
                )

            row = _require_cell_row(
                self.cells,
                frame=frame,
                cell_id=cell_id,
            )
            centroid = row[
                ["centroid_z", "centroid_y", "centroid_x"]
            ].to_numpy(dtype=float)
            bounds = _calculate_box_bounds(
                centroid_zyx=centroid,
                box_size_zyx=box_size,
                spatial_shape_zyx=self.raw_volume.shape[1:],
            )
            edges = _make_box_wireframe(
                frame=frame,
                requested_start_zyx=bounds["requested_start_zyx"],
                requested_stop_zyx=bounds["requested_stop_zyx"],
            )

            self.clear_preview(update_status=False)
            self._preview_layer = self.viewer.add_shapes(
                edges,
                shape_type="line",
                name=MERGED_CASE_PREVIEW_LAYER,
                edge_color="red",
                edge_width=3,
                opacity=1.0,
                scale=(1.0, *self.voxel_size_zyx),
            )
            try:
                self._preview_layer.mode = "pan_zoom"
            except (AttributeError, ValueError):
                pass

            self._preview_role = role
            padding_used = any(bounds["padding_before_zyx"]) or any(
                bounds["padding_after_zyx"]
            )
            padding_text = " Boundary padding will be used." if padding_used else ""
            self._set_status(
                f"Previewing {role}: frame {frame}, cell {cell_id}, "
                f"box {box_size}.{padding_text}"
            )

        except Exception as error:
            self._set_error(error)

    def clear_preview(
        self,
        *_args: Any,
        update_status: bool = True,
    ) -> None:
        if self._preview_layer is not None:
            try:
                self.viewer.layers.remove(self._preview_layer)
            except (ValueError, KeyError):
                pass
            finally:
                self._preview_layer = None
                self._preview_role = None

        if update_status:
            self._set_status("Preview cleared.")

    # --------------------------------------------------------
    # Capture workflow
    # --------------------------------------------------------

    def _capture_current_role(self, role: str) -> dict[str, Any]:
        frame = self.current_frame()
        cell_id = int(self.role_id_spins[role].value())
        box_size = self.current_box_size()

        if self._locked_box_size is not None and box_size != self._locked_box_size:
            raise ValueError(
                f"The current case uses box size {self._locked_box_size}. "
                "Reset the case before changing it."
            )

        return capture_cell_state(
            role=role,
            frame=frame,
            cell_id=cell_id,
            box_size_zyx=box_size,
            cells=self.cells,
            raw_volume=self.raw_volume,
            preprocessed_volume=self.preprocessed_volume,
            binary_mask_volume=self.binary_mask_volume,
            instance_labels_volume=self.instance_labels_volume,
            voxel_size_zyx=self.voxel_size_zyx,
            source_cell_files=self.source_cell_files,
            source_raw_zarr=self.source_raw_zarr,
            source_preprocessed_files=self.source_preprocessed_files,
            source_binary_mask_files=self.source_binary_mask_files,
            source_instance_label_files=self.source_instance_label_files,
        )

    def capture_parent(self, role: str) -> None:
        try:
            if role not in {"cell_1", "cell_2"}:
                raise ValueError(f"Unsupported parent role: {role}")

            capture = self._capture_current_role(role)

            other_role = "cell_2" if role == "cell_1" else "cell_1"
            if other_role in self._captures:
                current_key = (capture["frame"], capture["cell_id"])
                other = self._captures[other_role]
                other_key = (other["frame"], other["cell_id"])
                if current_key == other_key:
                    raise ValueError(
                        "Cell 1 and cell 2 cannot refer to the same cell in "
                        "the same frame."
                    )

            self._captures[role] = capture
            self._lock_box_size(capture["box_size_zyx"])
            self.role_status_labels[role].setText(
                f"Captured frame {capture['frame']}, cell {capture['cell_id']}, "
                f"box {capture['box_size_zyx']}"
            )
            self._set_status(
                f"Saved {role} in memory from frame {capture['frame']}. "
                "It will be written to disk by Save complete case."
            )
            show_info(
                f"Captured {role}: frame {capture['frame']}, "
                f"cell {capture['cell_id']}"
            )

        except Exception as error:
            self._set_error(error)

    def save_complete_case(self, *_args: Any) -> None:
        try:
            missing = [
                role
                for role in ("cell_1", "cell_2")
                if role not in self._captures
            ]
            if missing:
                raise ValueError(
                    "Capture both parent cells before saving the merged cell. "
                    f"Missing: {', '.join(missing)}"
                )

            merged_capture = self._capture_current_role("merged_cell")
            self._captures["merged_cell"] = merged_capture

            case_dir = save_multilevel_merged_cell_case(
                sample_id=self.sample_id,
                cell_1=self._captures["cell_1"],
                cell_2=self._captures["cell_2"],
                merged_cell=merged_capture,
                output_dir=self.output_dir,
                overwrite=self.overwrite_checkbox.isChecked(),
            )

            self._last_saved_dir = case_dir
            self.role_status_labels["merged_cell"].setText(
                f"Saved frame {merged_capture['frame']}, "
                f"cell {merged_capture['cell_id']}"
            )
            self._set_status(
                f"Complete merged-cell case saved to:\n{case_dir}\n"
                "Click Reset case before collecting another event."
            )
            show_info(f"Saved merged-cell case:\n{case_dir}")

        except Exception as error:
            self._set_error(error)

    def _lock_box_size(self, box_size: Sequence[int]) -> None:
        self._locked_box_size = _validate_box_size(box_size)
        for spin in (self.z_size_spin, self.y_size_spin, self.x_size_spin):
            spin.setEnabled(False)

    def _unlock_box_size(self) -> None:
        self._locked_box_size = None
        for spin in (self.z_size_spin, self.y_size_spin, self.x_size_spin):
            spin.setEnabled(True)

    def reset_case(self, *_args: Any) -> None:
        self._captures.clear()
        self._last_saved_dir = None
        self._unlock_box_size()
        self.clear_preview(update_status=False)

        for label in self.role_status_labels.values():
            label.setText("Not captured")

        self._set_status(
            "Case reset. Choose the shared box size, then capture cell 1, "
            "cell 2, and the merged cell from their respective frames."
        )

    # --------------------------------------------------------
    # Status
    # --------------------------------------------------------

    def _set_status(self, message: str) -> None:
        self.status_label.setStyleSheet("")
        self.status_label.setText(str(message))

    def _set_error(self, error: Exception) -> None:
        message = f"Merged-cell extraction failed: {error}"
        self.status_label.setStyleSheet("color: #d9534f;")
        self.status_label.setText(message)
        show_error(message)


def add_multilevel_merged_cell_extractor(
    *,
    viewer: Any,
    cells: pd.DataFrame,
    raw_volume: Any,
    preprocessed_volume: Any,
    binary_mask_volume: Any,
    instance_labels_volume: Any,
    sample_id: str,
    output_dir: Path = MERGED_CELLS_DIR,
    voxel_size_zyx: Sequence[float] = (1.625, 0.40625, 0.40625),
    default_box_size: Sequence[int] = DEFAULT_MERGED_CASE_BOX_SIZE,
    source_cell_files: Sequence[Path] | None = None,
    source_raw_zarr: Path | None = None,
    source_preprocessed_files: Sequence[Path] | None = None,
    source_binary_mask_files: Sequence[Path] | None = None,
    source_instance_label_files: Sequence[Path] | None = None,
) -> MultilevelMergedCellExtractorWidget:
    widget = MultilevelMergedCellExtractorWidget(
        viewer=viewer,
        cells=cells,
        raw_volume=raw_volume,
        preprocessed_volume=preprocessed_volume,
        binary_mask_volume=binary_mask_volume,
        instance_labels_volume=instance_labels_volume,
        sample_id=sample_id,
        output_dir=output_dir,
        voxel_size_zyx=voxel_size_zyx,
        default_box_size=default_box_size,
        source_cell_files=source_cell_files,
        source_raw_zarr=source_raw_zarr,
        source_preprocessed_files=source_preprocessed_files,
        source_binary_mask_files=source_binary_mask_files,
        source_instance_label_files=source_instance_label_files,
    )

    viewer.window.add_dock_widget(
        widget,
        name="Multilevel merged-cell extractor",
        area="right",
    )
    return widget


In [ ]:
import napari
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

VOXEL_SIZE = (
    1.625,      # Z
    0.40625,    # Y
    0.40625,    # X
)

SPATIAL_SHAPE_ZYX = original_volume.shape[-3:]


# ------------------------------------------------------------
# Find new and ended tracks
# ------------------------------------------------------------

first_frame = tracks["frame"].min()
last_frame = tracks["frame"].max()

track_summary = (
    tracks.groupby("track_id")
    .agg(
        first_frame=("frame", "min"),
        last_frame=("frame", "max"),
    )
    .reset_index()
)

# All apparent births and deaths before boundary filtering
all_new_track_ids = track_summary.loc[
    track_summary["first_frame"] > first_frame,
    "track_id",
]

all_ended_track_ids = track_summary.loc[
    track_summary["last_frame"] < last_frame,
    "track_id",
]


# ------------------------------------------------------------
# Endpoint extraction
# ------------------------------------------------------------

def get_track_endpoints(
        tracks_df,
        track_ids,
        endpoint,
):
    """
    Return the first or last detection of each selected track.

    Parameters
    ----------
    tracks_df : pandas.DataFrame
        Complete tracks DataFrame.

    track_ids : iterable
        IDs of the tracks whose endpoints should be extracted.

    endpoint : {"start", "end"}
        Which endpoint to extract.
    """

    selected = tracks_df[
        tracks_df["track_id"].isin(track_ids)
    ]

    if selected.empty:
        return selected.copy()

    grouped_frames = selected.groupby("track_id")["frame"]

    if endpoint == "start":
        row_indices = grouped_frames.idxmin()

    elif endpoint == "end":
        row_indices = grouped_frames.idxmax()

    else:
        raise ValueError(
            "endpoint must be either 'start' or 'end'"
        )

    return (
        selected.loc[row_indices]
        .sort_values("track_id")
        .reset_index(drop=True)
    )


# ------------------------------------------------------------
# Boundary classification
# ------------------------------------------------------------

def classify_boundary_endpoints(
        endpoint_rows,
        cells_df,
        spatial_shape_zyx,
        voxel_size_zyx,
        boundary_margin_um,
):
    """
    Determine whether each track endpoint is close to a volume boundary.

    The endpoint cell bounding box is used when available. If bounding-box
    information is unavailable, the endpoint centroid is used instead.
    """

    result = endpoint_rows.copy()

    if result.empty:
        result["boundary_distance_um"] = pd.Series(dtype=float)
        result["is_boundary_endpoint"] = pd.Series(dtype=bool)
        return result

    voxel_size = np.asarray(
        voxel_size_zyx,
        dtype=float,
    )

    spatial_shape = np.asarray(
        spatial_shape_zyx,
        dtype=float,
    )

    # --------------------------------------------------------
    # Centroid-based boundary distance
    # --------------------------------------------------------

    coordinates_zyx = result[
        ["z", "y", "x"]
    ].to_numpy(dtype=float)

    distance_to_lower_faces_um = (
            coordinates_zyx * voxel_size
    )

    distance_to_upper_faces_um = (
                                         spatial_shape - 1 - coordinates_zyx
                                 ) * voxel_size

    centroid_boundary_distance_um = np.minimum(
        distance_to_lower_faces_um,
        distance_to_upper_faces_um,
    ).min(axis=1)

    # --------------------------------------------------------
    # Prefer cell bounding-box distance when available
    # --------------------------------------------------------

    bbox_columns = [
        "z_min",
        "y_min",
        "x_min",
        "z_max",
        "y_max",
        "x_max",
    ]

    bbox_available = (
            "cell_id" in result.columns
            and "frame" in cells_df.columns
            and "cell_id" in cells_df.columns
            and set(bbox_columns).issubset(cells_df.columns)
    )

    if bbox_available:
        cell_lookup = (
            cells_df[
                [
                    "frame",
                    "cell_id",
                    *bbox_columns,
                ]
            ]
            .drop_duplicates(
                subset=["frame", "cell_id"]
            )
        )

        result = result.merge(
            cell_lookup,
            on=["frame", "cell_id"],
            how="left",
        )

        bbox_min_zyx = result[
            ["z_min", "y_min", "x_min"]
        ].to_numpy(dtype=float)

        bbox_max_zyx = result[
            ["z_max", "y_max", "x_max"]
        ].to_numpy(dtype=float)

        bbox_distance_to_lower_faces_um = (
                bbox_min_zyx * voxel_size
        )

        # z_max, y_max and x_max are treated as exclusive bounds.
        bbox_distance_to_upper_faces_um = (
                                                  spatial_shape - bbox_max_zyx
                                          ) * voxel_size

        bbox_boundary_distance_um = np.minimum(
            bbox_distance_to_lower_faces_um,
            bbox_distance_to_upper_faces_um,
        ).min(axis=1)

        valid_bbox = np.isfinite(
            bbox_boundary_distance_um
        )

        result["boundary_distance_um"] = np.where(
            valid_bbox,
            bbox_boundary_distance_um,
            centroid_boundary_distance_um,
        )

    else:
        result["boundary_distance_um"] = (
            centroid_boundary_distance_um
        )

    result["is_boundary_endpoint"] = (
            result["boundary_distance_um"]
            <= boundary_margin_um
    )

    return result


# ------------------------------------------------------------
# Extract relevant endpoints
# ------------------------------------------------------------

new_track_endpoints = get_track_endpoints(
    tracks_df=tracks,
    track_ids=all_new_track_ids,
    endpoint="start",
)

ended_track_endpoints = get_track_endpoints(
    tracks_df=tracks,
    track_ids=all_ended_track_ids,
    endpoint="end",
)


# ------------------------------------------------------------
# Classify endpoints as boundary or non-boundary
# ------------------------------------------------------------

new_track_endpoints = classify_boundary_endpoints(
    endpoint_rows=new_track_endpoints,
    cells_df=cells_df,
    spatial_shape_zyx=SPATIAL_SHAPE_ZYX,
    voxel_size_zyx=VOXEL_SIZE,
    boundary_margin_um=BOUNDARY_MARGIN_UM,
)

ended_track_endpoints = classify_boundary_endpoints(
    endpoint_rows=ended_track_endpoints,
    cells_df=cells_df,
    spatial_shape_zyx=SPATIAL_SHAPE_ZYX,
    voxel_size_zyx=VOXEL_SIZE,
    boundary_margin_um=BOUNDARY_MARGIN_UM,
)


# ------------------------------------------------------------
# Separate failure candidates from boundary entries/exits
# ------------------------------------------------------------

boundary_new_track_ids = new_track_endpoints.loc[
    new_track_endpoints["is_boundary_endpoint"],
    "track_id",
]

boundary_ended_track_ids = ended_track_endpoints.loc[
    ended_track_endpoints["is_boundary_endpoint"],
    "track_id",
]

new_track_ids = new_track_endpoints.loc[
    ~new_track_endpoints["is_boundary_endpoint"],
    "track_id",
]

ended_track_ids = ended_track_endpoints.loc[
    ~ended_track_endpoints["is_boundary_endpoint"],
    "track_id",
]


# ------------------------------------------------------------
# Build track DataFrames
# ------------------------------------------------------------

# Non-boundary failure candidates
new_tracks = tracks[
    tracks["track_id"].isin(new_track_ids)
].copy()

ended_tracks = tracks[
    tracks["track_id"].isin(ended_track_ids)
].copy()

# Boundary entries and exits
boundary_new_tracks = tracks[
    tracks["track_id"].isin(boundary_new_track_ids)
].copy()

boundary_ended_tracks = tracks[
    tracks["track_id"].isin(boundary_ended_track_ids)
].copy()


# ------------------------------------------------------------
# Launch Napari
# ------------------------------------------------------------

viewer = napari.Viewer(ndisplay=3)


# ------------------------------------------------------------
# Failure-analysis layers
# ------------------------------------------------------------

viewer.add_image(
    original_volume,
    name="Raw Volume",
    scale=(1, *VOXEL_SIZE),
    rendering="mip",
    colormap="gray",
    contrast_limits=[
        np.percentile(original_volume, 1),
        np.percentile(original_volume, 99.8),
    ],
)


# ------------------------------------------------------------
# Ended failure candidates
# ------------------------------------------------------------

if not ended_tracks.empty:
    viewer.add_tracks(
        ended_tracks[
            ["track_id", "frame", "z", "y", "x"]
        ].to_numpy(float),
        name="Ended Tracks",
        scale=(1, *VOXEL_SIZE),
        tail_length=20,
    )

    viewer.add_points(
        ended_tracks[
            ["frame", "z", "y", "x"]
        ].to_numpy(float),
        name="Ended Centroids",
        scale=(1, *VOXEL_SIZE),
        size=4,
        face_color="red",
    )


# ------------------------------------------------------------
# New failure candidates
# ------------------------------------------------------------

if not new_tracks.empty:
    viewer.add_tracks(
        new_tracks[
            ["track_id", "frame", "z", "y", "x"]
        ].to_numpy(float),
        name="New Tracks",
        scale=(1, *VOXEL_SIZE),
        tail_length=20,
    )

    viewer.add_points(
        new_tracks[
            ["frame", "z", "y", "x"]
        ].to_numpy(float),
        name="New Centroids",
        scale=(1, *VOXEL_SIZE),
        size=4,
        face_color="lime",
    )


# ------------------------------------------------------------
# Optional boundary tracks
# ------------------------------------------------------------

if SHOW_BOUNDARY_TRACKS:

    # Tracks entering through a volume boundary
    if not boundary_new_tracks.empty:
        viewer.add_tracks(
            boundary_new_tracks[
                ["track_id", "frame", "z", "y", "x"]
            ].to_numpy(float),
            name="Boundary Entry Tracks",
            scale=(1, *VOXEL_SIZE),
            tail_length=20,
        )

        viewer.add_points(
            boundary_new_tracks[
                ["frame", "z", "y", "x"]
            ].to_numpy(float),
            name="Boundary Entry Centroids",
            scale=(1, *VOXEL_SIZE),
            size=4,
            face_color="cyan",
        )

    # Tracks leaving through a volume boundary
    if not boundary_ended_tracks.empty:
        viewer.add_tracks(
            boundary_ended_tracks[
                ["track_id", "frame", "z", "y", "x"]
            ].to_numpy(float),
            name="Boundary Exit Tracks",
            scale=(1, *VOXEL_SIZE),
            tail_length=20,
        )

        viewer.add_points(
            boundary_ended_tracks[
                ["frame", "z", "y", "x"]
            ].to_numpy(float),
            name="Boundary Exit Centroids",
            scale=(1, *VOXEL_SIZE),
            size=4,
            face_color="orange",
        )


# ------------------------------------------------------------
# All original tracks
# ------------------------------------------------------------

viewer.add_image(
    original_volume,
    name="Raw Volume - all",
    rendering="mip",
    colormap="gray",
    contrast_limits=[
        np.percentile(original_volume, 1),
        np.percentile(original_volume, 99.8),
    ],
    scale=(1, *VOXEL_SIZE),
).visible = False

viewer.add_tracks(
    tracks_array,
    name="Tracks - all",
    tail_length=20,
    scale=(1, *VOXEL_SIZE),
).visible = False

viewer.add_points(
    points_array,
    name="Centroids - all",
    size=4,
    face_color="red",
    properties={
        "cell_id": tracks["cell_id"].to_numpy(),
        "track_id": track_ids,
    },
    scale=(1, *VOXEL_SIZE),
    text={
        "string": "{cell_id}",
        "size": 8,
        "color": "white",
        "anchor": "center",
    },
).visible = False


# ------------------------------------------------------------
# Diagnostic summary
# ------------------------------------------------------------

print(
    f"New tracks before boundary filtering: "
    f"{len(all_new_track_ids)}"
)

print(
    f"Boundary-entry tracks: "
    f"{len(boundary_new_track_ids)}"
)

print(
    f"New failure candidates: "
    f"{len(new_track_ids)}"
)

print()

print(
    f"Ended tracks before boundary filtering: "
    f"{len(all_ended_track_ids)}"
)

print(
    f"Boundary-exit tracks: "
    f"{len(boundary_ended_track_ids)}"
)

print(
    f"Ended failure candidates: "
    f"{len(ended_track_ids)}"
)

print()

print(
    f"SHOW_BOUNDARY_TRACKS: "
    f"{SHOW_BOUNDARY_TRACKS}"
)


# ------------------------------------------------------------
# Multilevel merged-cell extraction tool
# ------------------------------------------------------------

merged_case_extractor = add_multilevel_merged_cell_extractor(
    viewer=viewer,
    cells=cells_df,
    raw_volume=original_volume,
    preprocessed_volume=preprocessed_volume,
    binary_mask_volume=binary_mask_volume,
    instance_labels_volume=instance_labels_volume,
    sample_id=SAMPLE_ID,
    output_dir=MERGED_CELLS_DIR,
    voxel_size_zyx=VOXEL_SIZE,
    default_box_size=DEFAULT_MERGED_CASE_BOX_SIZE,
    source_cell_files=cell_files,
    source_raw_zarr=ARRAY_PATH,
    source_preprocessed_files=preprocessing_files,
    source_binary_mask_files=masking_files,
    source_instance_label_files=segmentation_files,
)

napari.run()
